In [ ]:
import pandas as pd
df = pd.read_csv('../data/tedsd_puf_2023.csv'))
print(df.shape)
print(df.dtypes)
df.head()
#check reason column
print(df['REASON'].value_counts().sort_index())


In [ ]:
# Create binary completion column
df['completed'] = (df['REASON'] == 1).astype(int)

# Verify it worked
print(df['completed'].value_counts())
print(f"\nOverall completion rate: {df['completed'].mean():.1%}")

In [ ]:
import numpy as np

# Replace all -9 missing value codes with NaN
df.replace(-9, np.nan, inplace=True)

# Check how many missing values we now have per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)

missing_summary = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
})

print(missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_pct', ascending=False))

In [ ]:
# Select only the columns we need
cols = ['DISYR', 'REASON', 'completed', 'SUB1', 
        'AGE', 'SEX', 'RACE', 'LOS', 'SERVICES']

df_clean = df[cols].copy()

print(df_clean.shape)
print(df_clean.isnull().sum())
df_clean.head()

In [ ]:
# Define mappings from codebook
sub1_map = {
    1: 'None', 2: 'Alcohol', 3: 'Cocaine/Crack',
    4: 'Marijuana', 5: 'Heroin', 6: 'Methadone',
    7: 'Other Opiates', 8: 'PCP', 9: 'Hallucinogens',
    10: 'Methamphetamine', 11: 'Other Amphetamines',
    12: 'Other Stimulants', 13: 'Benzodiazepines',
    14: 'Other Tranquilizers', 15: 'Barbiturates',
    16: 'Other Sedatives', 17: 'Inhalants',
    18: 'OTC Medications', 19: 'Other Drugs'
}

age_map = {
    1: '12-14', 2: '15-17', 3: '18-20', 4: '21-24',
    5: '25-29', 6: '30-34', 7: '35-39', 8: '40-44',
    9: '45-49', 10: '50-54', 11: '55-64', 12: '65+'
}

sex_map = {1: 'Male', 2: 'Female'}

race_map = {
    1: 'Alaska Native', 2: 'American Indian',
    3: 'Asian or Pacific Islander', 4: 'Black or African American',
    5: 'White', 6: 'Asian', 7: 'Other',
    8: 'Two or More Races', 9: 'Native Hawaiian or Pacific Islander'
}

# Apply mappings
df_clean['SUB1'] = df_clean['SUB1'].map(sub1_map)
df_clean['AGE'] = df_clean['AGE'].map(age_map)
df_clean['SEX'] = df_clean['SEX'].map(sex_map)
df_clean['RACE'] = df_clean['RACE'].map(race_map)

# Confirm it worked
df_clean.head()

In [ ]:
df_clean.to_csv('../data/tedsd_clean.csv', index=False)
print("Clean file saved successfully")
print(f"Final shape: {df_clean.shape}")

# 01 - Data Cleaning

This notebook loads the raw SAMHSA TEDS-D 2023 dataset (1,474,025 discharge records), 
replaces SAMHSA missing value codes (-9) with NaN, selects the 9 variables relevant 
to this analysis, decodes numeric codes into readable labels using the official codebook, 
and exports a cleaned dataset for analysis.

Key decisions:
- Columns with >50% missing data were excluded from analysis
- SUB1 (10.1% missing) retained but missing rows excluded during substance-level analysis
- REASON binarized into a completion indicator (1 = completed, 0 = all other outcomes)